# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [ ]:
# =====================================================
# TUGAS 1 ANALISIS BIG DATA
# MILESTONE 1: DATA PROFILING
# DATASET AGROKLIMAT INDONESIA
# =====================================================

# 1. IMPORT LIBRARY
from pathlib import Path
import polars as pl
import duckdb
import pandas as pd

# 2. MENENTUKAN LOKASI PROYEK
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# 3. MENENTUKAN LOKASI DATASET
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "agroclimatic_indonesia"
    / "data"
    / "processed"
    / "agroclimate_with_indicators.parquet"
)

# 4. MEMERIKSA DATASET
print("=" * 50)
print("PEMERIKSAAN DATASET")
print("=" * 50)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di lokasi: {DATA_PATH}"
    )

print("Status       : Dataset berhasil ditemukan")
print("Nama file    :", DATA_PATH.name)
print("Ukuran file  :", round(DATA_PATH.stat().st_size / 1_000_000, 2), "MB")


# =====================================================
# 5. PROFILING MENGGUNAKAN POLARS LAZYFRAME
# =====================================================

print("\n" + "=" * 50)
print("PROFILING MENGGUNAKAN POLARS")
print("=" * 50)

# Membaca dataset menggunakan LazyFrame
lazy_data = pl.scan_parquet(DATA_PATH)

# Melihat struktur dataset
print("\nSTRUKTUR KOLOM:")
print(lazy_data.collect_schema())

# Menghitung jumlah baris dan kolom
jumlah_baris = lazy_data.select(
    pl.len().alias("jumlah_baris")
).collect()

jumlah_kolom = len(lazy_data.collect_schema())

print("\nJUMLAH BARIS:")
display(jumlah_baris)

print("JUMLAH KOLOM:", jumlah_kolom)

# Menghitung missing values
print("\nMISSING VALUES:")

missing_values = lazy_data.select(
    pl.all().null_count()
).collect()

display(missing_values)

# Menampilkan 10 data pertama
print("\n10 DATA PERTAMA:")

contoh_data = lazy_data.head(10).collect()

display(contoh_data)


# =====================================================
# 6. PROFILING MENGGUNAKAN DUCKDB
# =====================================================

print("\n" + "=" * 50)
print("PROFILING MENGGUNAKAN DUCKDB")
print("=" * 50)

# Membuat koneksi DuckDB
connection = duckdb.connect()

# Menjalankan profiling menggunakan SUMMARIZE
profil_duckdb = connection.execute(
    """
    SUMMARIZE
    SELECT *
    FROM read_parquet(?)
    """,
    [str(DATA_PATH)]
).fetchdf()

# Menampilkan hasil profiling
display(profil_duckdb)

print("\nProfiling DuckDB berhasil!")

PEMERIKSAAN DATASET
Status       : Dataset berhasil ditemukan
Nama file    : agroclimate_with_indicators.parquet
Ukuran file  : 60.01 MB

PROFILING MENGGUNAKAN POLARS

STRUKTUR KOLOM:
Schema([('grid_id', String), ('latitude', Float64), ('longitude', Float64), ('date', Datetime(time_unit='ns', time_zone=None)), ('temperature_2m_mean', Float64), ('temperature_2m_max', Float64), ('temperature_2m_min', Float64), ('precipitation_sum', Float64), ('relative_humidity_2m_mean', Int64), ('et0_fao_evapotranspiration', Float64), ('qc_range_flag', Boolean), ('rainfall_7d', Float64), ('rainfall_30d', Float64), ('rainfall_90d', Float64), ('rainy_days_30d', Float64), ('rainy_days_90d', Float64), ('max_daily_rainfall_30d', Float64), ('consecutive_dry_days', Int64), ('temperature_range', Float64), ('rolling_temperature_mean_7d', Float64), ('cumulative_et0_30d', Float64), ('water_balance', Float64), ('rolling_water_balance_30d', Float64)])

JUMLAH BARIS:


jumlah_baris
u32
2235636


JUMLAH KOLOM: 23

MISSING VALUES:


grid_id,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean,et0_fao_evapotranspiration,qc_range_flag,rainfall_7d,rainfall_30d,rainfall_90d,rainy_days_30d,rainy_days_90d,max_daily_rainfall_30d,consecutive_dry_days,temperature_range,rolling_temperature_mean_7d,cumulative_et0_30d,water_balance,rolling_water_balance_30d
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



10 DATA PERTAMA:


grid_id,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean,et0_fao_evapotranspiration,qc_range_flag,rainfall_7d,rainfall_30d,rainfall_90d,rainy_days_30d,rainy_days_90d,max_daily_rainfall_30d,consecutive_dry_days,temperature_range,rolling_temperature_mean_7d,cumulative_et0_30d,water_balance,rolling_water_balance_30d
str,f64,f64,datetime[ns],f64,f64,f64,f64,i64,f64,bool,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64
"""GRID_+00000_+100000""",0.0,100.0,2016-01-01 00:00:00,23.1,24.1,21.9,28.0,95,1.11,false,28.0,28.0,28.0,1.0,1.0,28.0,0,2.2,23.1,1.11,26.89,26.89
"""GRID_+00000_+100000""",0.0,100.0,2016-01-02 00:00:00,23.1,26.5,21.4,14.2,93,2.35,false,42.2,42.2,42.2,2.0,2.0,28.0,0,5.1,23.1,3.46,11.85,38.74
"""GRID_+00000_+100000""",0.0,100.0,2016-01-03 00:00:00,24.2,29.8,20.0,1.9,88,4.3,false,44.1,44.1,44.1,3.0,3.0,28.0,0,9.8,23.466667,7.76,-2.4,36.34
"""GRID_+00000_+100000""",0.0,100.0,2016-01-04 00:00:00,24.0,28.2,21.6,8.6,91,3.55,false,52.7,52.7,52.7,4.0,4.0,28.0,0,6.6,23.6,11.31,5.05,41.39
"""GRID_+00000_+100000""",0.0,100.0,2016-01-05 00:00:00,24.0,28.5,21.7,4.7,91,3.65,false,57.4,57.4,57.4,5.0,5.0,28.0,0,6.8,23.68,14.96,1.05,42.44
"""GRID_+00000_+100000""",0.0,100.0,2016-01-06 00:00:00,24.4,28.9,22.0,5.0,89,3.7,false,62.4,62.4,62.4,6.0,6.0,28.0,0,6.9,23.8,18.66,1.3,43.74
"""GRID_+00000_+100000""",0.0,100.0,2016-01-07 00:00:00,24.2,28.2,22.0,7.7,89,3.46,false,70.1,70.1,70.1,7.0,7.0,28.0,0,6.2,23.857143,22.12,4.24,47.98
"""GRID_+00000_+100000""",0.0,100.0,2016-01-08 00:00:00,24.5,29.3,21.3,6.0,85,4.2,false,48.1,76.1,76.1,8.0,8.0,28.0,0,8.0,24.057143,26.32,1.8,49.78
"""GRID_+00000_+100000""",0.0,100.0,2016-01-09 00:00:00,23.9,28.6,21.7,15.5,90,3.19,false,49.4,91.6,91.6,9.0,9.0,28.0,0,6.9,24.171429,29.51,12.31,62.09



PROFILING MENGGUNAKAN DUCKDB


AttributeError: 'DuckDBPyConnection' object has no attribute 'pl'

## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

In [ ]:
contoh_data = lazy_data.head(10).collect()
contoh_data

## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
).pl()
profil_duckdb

## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.